# M08.3 — RL smoke run: GRPO + LoRA on Qwen3.5-2B against the frozen D1 detector

**Thin launcher.** All logic lives in the repo behind a CLI:

    python -m pii_redteam.cli rl-smoke --config configs/m08_rl_smoke.yaml

This is a **capped smoke run, not the pilot**. It answers: does the TRL update path work on the
real model (4-bit QLoRA, T4 memory and speed), and is there reward signal for GRPO to use?

1. Samples 8 rollouts for each of 32 held-out requests with the untrained model and scores
   them with frozen D1 (**pre-RL signal**: valid rate, how many groups contain a detector miss).
2. Trains 30 GRPO steps (4 prompts x 8 rollouts per step, `scale_rewards=False`, `beta=0.02`).
3. Samples the same held-out requests again (**post-RL**) and saves every raw output.

**Hard caps:** 30 steps and `max_minutes: 60` of training wall clock (then save and stop).
Checkpoints every 10 steps go to Drive.

**Before you run (owner):** confirm the cost cap; put the frozen D1 folder
`d1-frozen-d1v2-20261110-075731/` (sha-verified) and the AI4Privacy cache
(`cache/1english_openpii_30k.jsonl`) under `MyDrive/pii-redteam-rl/`; make sure `REF` below
exists on GitHub (Colab clones from there). **Shut the runtime down when it finishes** (last cell).

**Bring back:** `rl_smoke_report.json`, `samples_pre.jsonl`, `samples_post.jsonl`.
Smoke checks are health checks, not learning claims; read the samples by hand for fluency
and diversity before anything else.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
REF = "feat/m08-rl-smoke"  # branch holding the RL code; change it if you named it differently
D1_DIR = os.path.join(DRIVE_ROOT, "d1-frozen-d1v2-20261110-075731")
CACHE = os.path.join(DRIVE_ROOT, "cache")

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Always sync to the target ref: a reused VM may hold a STALE clone.
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "fetch", "-q", "origin", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-q", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", REF, REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

for path in (os.path.join(D1_DIR, "model.safetensors"), os.path.join(CACHE, "1english_openpii_30k.jsonl")):
    assert os.path.exists(path), f"missing on Drive: {path}"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "peft",
     "bitsandbytes", "datasets", "pyyaml", "trl>=1.17"],
    check=True,
)

import torch

assert torch.cuda.is_available(), "GPU runtime required."
free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb >= 10, (
    f"GPU has only {free_gb:.1f} GB free - dirty runtime. Runtime -> Disconnect and"
    " delete runtime, then reopen from GitHub and Run all."
)
print(f"GPU free: {free_gb:.1f} GB - OK ({torch.cuda.get_device_name(0)})")

In [ ]:
import yaml

env = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src"))

# Rebuild the locked splits and name pools (asserts the final-eval lock inside the run).
subprocess.run(
    [sys.executable, "scripts/carve_splits.py", "--cache-dir", CACHE,
     "--out", os.path.join(REPO_DIR, "artifacts", "splits")],
    cwd=REPO_DIR, env=env, check=True,
)

with open(os.path.join(REPO_DIR, "configs/m08_rl_smoke.yaml"), encoding="utf-8") as f:
    config = yaml.safe_load(f)
config["work_dir"] = DRIVE_ROOT
config["detector"]["path"] = D1_DIR
colab_config = os.path.join(REPO_DIR, "configs/m08_rl_smoke_colab.yaml")
with open(colab_config, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Running the smoke run (pre-RL sampling, 30 GRPO steps, post-RL sampling)...")
result = subprocess.run(
    [sys.executable, "-m", "pii_redteam.cli", "rl-smoke", "--config", "configs/m08_rl_smoke_colab.yaml"],
    cwd=REPO_DIR, env=dict(env, PYTHONUNBUFFERED="1"),
)
print(f"\nsmoke run exit code: {result.returncode}")
if result.returncode != 0:
    print("^^ the traceback above is the real error - paste it back to me")

In [ ]:
import glob
import json
import random

runs = sorted(glob.glob(os.path.join(DRIVE_ROOT, "m08-smoke-*")))
run_dir = runs[-1]
print("run dir:", run_dir)
rng = random.Random(0)
for stage in ("pre", "post"):
    with open(os.path.join(run_dir, f"samples_{stage}.jsonl"), encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    valid = [r for r in rows if r["status"] == "valid"]
    print(f"\n=== {stage.upper()}-RL: {len(valid)}/{len(rows)} valid; 6 highest-reward and 6 random valid ===")
    top = sorted(valid, key=lambda r: -r["reward"])[:6]
    for r in top + rng.sample(valid, min(6, len(valid))):
        print(f"[{r['request_id']} reward={r['reward']:.3f}] {r['raw_text'][:240]!r}")

**When the run finishes:** the PASS/FAIL list is printed above and the review cell shows the
highest-reward and random valid samples before and after training. Read the highest-reward
ones first: that is where reward hacking or odd text shows up.
Then run the next cell to release the GPU.

In [ ]:
from google.colab import runtime

runtime.unassign()